# Tutorial 1: Environments and Neural Networks

**zrth** extracts symbolic reactive modules from Python classes. You write a standard [Gymnasium](https://gymnasium.farama.org/) environment or a [PyTorch](https://pytorch.org/) neural network, then wrap it with `Env()` or `NN()` — zrth analyzes it and produces a formal model with typed wires and terms that can be used for verification or training.

This tutorial walks through:
1. Defining an environment (plain `gym.Env`) and wrapping it with `Env()`
2. Defining a neural network (plain `nn.Module`) and wrapping it with `NN()`
3. Inspecting the extracted modules

## Defining an environment

Write a standard [Gymnasium](https://gymnasium.farama.org/) environment by subclassing `gym.Env`:

- **`__init__`**: set `action_space`, `observation_space`, any parameters (e.g. `self.y0`), and state variables (e.g. `self.x`)
- **`reset`**: initialize all state variables, return `(observation, info)`
- **`step`**: update state given an action, return `(observation, reward, terminated, truncated, info)`

**Important:** state variables must also be assigned in `__init__` so the analyzer can infer their types. For example, even though `reset` sets `self.x = 0.0`, you should also write `self.x = 0.0` in `__init__`.

Below is a counter system with three variables $x$, $y$, $z$. The variable $x$ increments while $x < y \lor x < z$, and resets to $0$ otherwise. We want to show that $x = y \lor x = z$ must occur infinitely often.

In [1]:
import torch
import torch.nn as nn
import gymnasium as gym
from gymnasium import spaces
from zrth import Env, NN


class CounterEnv(gym.Env):
    """Counter environment: x increments while x < y or x < z, resets otherwise."""

    def __init__(self, y0=5.0, z0=3.0):
        super().__init__()

        self.action_space = spaces.Discrete(1)
        self.observation_space = spaces.Box(low=-1e6, high=1e6, shape=(1,))

        self.y0 = y0
        self.z0 = z0

        # Also set state variables here so the analyzer can infer their dtype
        self.x = 0.0
        self.y = y0
        self.z = z0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.x = 0.0
        self.y = self.y0
        self.z = self.z0

        observation = self.x
        return observation, {}

    def step(self, action):
        if self.x < self.y or self.x < self.z:
            self.x = self.x + 1.0
        else:
            self.x = 0.0

        # y and z remain constant
        self.y = self.y
        self.z = self.z

        at_target = self.x == self.y or self.x == self.z
        reward = 1.0 if at_target else 0.0
        terminated = at_target
        truncated = False
        observation = self.x
        return observation, reward, terminated, truncated, {}

## Wrapping and inspection

When you call `Env(counter_instance)`, zrth analyzes the class's `__init__`, `reset`, and `step` methods. It:
1. Infers action/observation types from the gymnasium spaces
2. Classifies `self.*` attributes as **private** (read+write state) or **parameters** (read-only constants)
3. Creates typed wire pairs for each variable and signal
4. Converts `reset` into **init** terms and `step` into **update** terms

Print the extracted module to see the result.

In [2]:
counter = CounterEnv(y0=5.0, z0=3.0)
wrapped_counter = Env(counter)
print(wrapped_counter)

module
  external
    w8, w9 : Float(1)
  interface
    w10, w11 : Float(1)
    w12, w13 : Float(1)
    w14, w15 : Bool(1)
    w16, w17 : Bool(1)
  private
    w0, w1 : Float(1)
    w2, w3 : Float(1)
    w4, w5 : Float(1)
  atom controls w1, w3, w5, w11, w13, w15, w17 reads w0, w2, w4
  init
    Tensor([5]) w6; 
    Tensor([3]) w7; 
    Tensor([0]) w18; 
    Id w3; w18
    Id w5; w6
    Id w1; w7
    Id w11; w18
    Tensor([0]) w13; 
    Const: false w15; 
    Const: false w17; 
  update
    Tensor([5]) w6; 
    Tensor([3]) w7; 
    Lt w19; w2, w4
    Lt w20; w2, w0
    Tensor([0]) w21; 
    Tensor([1]) w22; 
    Ite w23; w19, w22, w20
    Tensor([1]) w24; 
    Add w25; w2, w24
    Tensor([0]) w26; 
    Ite w27; w23, w25, w26
    Id w3; w27
    Id w5; w4
    Id w1; w0
    Eq w28; w27, w4
    Eq w29; w27, w0
    Tensor([0]) w30; 
    Tensor([1]) w31; 
    Ite w32; w28, w31, w29
    Tensor([1]) w33; 
    Tensor([0]) w34; 
    Ite w35; w32, w33, w34
    Tensor([0]) w36; 
    Id w11; w27
 

## Reading the output

The printed module shows:

- **external**: input wires — here, the action (unused by this environment, but still present)
- **interface**: output wires — observation, reward, terminated, truncated
- **private**: internal state — wire pairs `wN, wM` for x, y, z (each pair is `latched, next`)
- **init**: terms from `reset` — how each wire is initialized
- **update**: terms from `step` — how each wire is updated, including `Ite` (if-then-else), `Lt`, `Eq`, `Add`

## Defining a neural network

Write a standard `nn.Module`, then wrap it with `NN(instance)` to extract a symbolic module:

- **`__init__`**: define `nn.Linear` layers
- **`forward`**: standard PyTorch forward pass

zrth extracts the architecture as a **combinatorial** (stateless) module. The resulting module has:
- **extl** (external input) wire pair — sized from the first layer's `in_features`
- **intf** (interface output) wire pair — sized from the last layer's `out_features`

The ranking function below maps $(x, y, z) \in \mathbb{R}^3$ to a scalar. It is used in formal verification to prove that the liveness property $x = y \lor x = z$ holds infinitely often — not a controller, so we don't compose it with the counter.

In [3]:
class RankingNN(nn.Module):
    """Ranking function: R(x, y, z) -> scalar.

    Architecture: [3] -> 2 -> [1]
    """

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3, 2)
        self.fc2 = nn.Linear(2, 1)

    def forward(self, state):
        x = torch.relu(self.fc1(state))
        return self.fc2(x)

Wrap and inspect. Note that the NN module has no `private` wires (it's stateless) and the `init`/`update` blocks are identical (combinatorial — same computation every step).

In [4]:
ranking = RankingNN()
wrapped_ranking = NN(ranking)
print(wrapped_ranking)

module
  external
    w37, w38 : Float(3)
  interface
    w39, w40 : Float(1)
  atom controls w40 awaits w38
  init
    Tensor([0.0051712049171328545 0.46728965640068054 0.024666307494044304 -0.1532318890094757 -0.1060895025730133 ...]) w41; 
    Tensor([0.04366765543818474 0.41038838028907776]) w42; 
    Linear w43; w38, w41, w42
    ReLU w44; w43
    Tensor([0.6814000606536865 -0.10748204588890076]) w45; 
    Tensor([-0.6033737063407898]) w46; 
    Linear w47; w44, w45, w46
    Id w40; w47
  update
    Tensor([0.0051712049171328545 0.46728965640068054 0.024666307494044304 -0.1532318890094757 -0.1060895025730133 ...]) w41; 
    Tensor([0.04366765543818474 0.41038838028907776]) w42; 
    Linear w43; w38, w41, w42
    ReLU w44; w43
    Tensor([0.6814000606536865 -0.10748204588890076]) w45; 
    Tensor([-0.6033737063407898]) w46; 
    Linear w47; w44, w45, w46
    Id w40; w47



## Training the ranking function

A **ranking function** $R: \mathcal{S} \to \mathbb{R}$ can be used to prove that a liveness property holds infinitely often. The key conditions are:

1. $R(s) \geq 0$ for all states $s$
2. $R(s) = 0$ when the liveness property holds (here: $x = y \lor x = z$)
3. $R(s') < R(s)$ whenever the property does *not* hold (the ranking strictly decreases)

If such an $R$ exists, the property must hold infinitely often — otherwise $R$ would decrease forever below zero, contradicting condition 1.

We train the `RankingNN` to approximate these conditions by:
1. Simulating the counter for $N$ steps (it's a fully functional `gym.Env`)
2. At each step, computing $R(x, y, z)$ and $R(x', y', z')$
3. Penalizing violations: $R$ should decrease at non-target states and be near zero at target states

In [5]:
optimizer = torch.optim.Adam(wrapped_ranking.parameters(), lr=0.01)
margin = 0.1
n_epochs = 300
n_steps = 20

for epoch in range(n_epochs):
    # Simulate the counter to collect a trajectory of (x, y, z) states.
    # CounterEnv is a full gym.Env — reset() and step() work directly.
    wrapped_counter.reset()

    states = []
    for _ in range(n_steps):
        states.append((wrapped_counter.x, wrapped_counter.y, wrapped_counter.z))
        wrapped_counter.step(0)  # Discrete(1) action space — always 0

    # Compute ranking loss over consecutive state pairs
    loss = torch.tensor(0.0)
    for i in range(len(states) - 1):
        s = torch.tensor(states[i], dtype=torch.float32)
        s_next = torch.tensor(states[i + 1], dtype=torch.float32)
        r = wrapped_ranking(s.unsqueeze(0)).squeeze()
        r_next = wrapped_ranking(s_next.unsqueeze(0)).squeeze()

        x, y, z = states[i]
        at_target = (x == y) or (x == z)

        if not at_target:
            # R must strictly decrease by at least `margin`
            loss = loss + torch.relu(r_next - r + margin)
            # R must be non-negative
            loss = loss + torch.relu(-r)
        else:
            # R should be ~0 at target states
            loss = loss + r ** 2

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"epoch {epoch:3d}  loss = {loss.item():.4f}")

epoch   0  loss = 8.4956
epoch  50  loss = 0.0931
epoch 100  loss = 0.0725
epoch 150  loss = 0.0634
epoch 200  loss = 0.0653
epoch 250  loss = 0.0634


## Evaluating the trained ranking function

After training, we evaluate $R$ along a trajectory. At non-target states, $R$ should decrease on each step. At target states ($x = y$ or $x = z$), $R$ should be near zero.

In [6]:
wrapped_counter.reset()

print(f"{'step':>4}  {'x':>5} {'y':>5} {'z':>5}  {'R(s)':>8}  {'target':>6}")
print("-" * 45)

with torch.no_grad():
    for step in range(20):
        x, y, z = wrapped_counter.x, wrapped_counter.y, wrapped_counter.z
        s = torch.tensor([x, y, z], dtype=torch.float32)
        r = wrapped_ranking(s.unsqueeze(0)).squeeze().item()
        at_target = (x == y) or (x == z)
        print(f"{step:4d}  {x:5.1f} {y:5.1f} {z:5.1f}  {r:8.4f}  {'  *' if at_target else ''}")
        wrapped_counter.step(0)

step      x     y     z      R(s)  target
---------------------------------------------
   0    0.0   5.0   3.0    0.4052  
   1    1.0   5.0   3.0    0.3050  
   2    2.0   5.0   3.0    0.2048  
   3    3.0   5.0   3.0    0.1045    *
   4    4.0   5.0   3.0    0.0043  
   5    5.0   5.0   3.0   -0.0959    *
   6    0.0   5.0   3.0    0.4052  
   7    1.0   5.0   3.0    0.3050  
   8    2.0   5.0   3.0    0.2048  
   9    3.0   5.0   3.0    0.1045    *
  10    4.0   5.0   3.0    0.0043  
  11    5.0   5.0   3.0   -0.0959    *
  12    0.0   5.0   3.0    0.4052  
  13    1.0   5.0   3.0    0.3050  
  14    2.0   5.0   3.0    0.2048  
  15    3.0   5.0   3.0    0.1045    *
  16    4.0   5.0   3.0    0.0043  
  17    5.0   5.0   3.0   -0.0959    *
  18    0.0   5.0   3.0    0.4052  
  19    1.0   5.0   3.0    0.3050  
